# SQL Queries: Middle Income Trap Analysis
**BUS 32120 Final Project | Winter 2026**

This file contains all 12 SQL queries used in our analysis. We use **SQLite in Python** to query 4 normalized tables:
1. `countries` — country metadata and group labels
2. `econ_indicators` — core macroeconomic data (15 indicators, from World Bank API)
3. `governance` — institutional quality scores (from World Bank API, 1996+)
4. `innovation` — R&D and technology indicators (from World Bank API)

Each query includes **WHAT** it does, **HOW** it works, and **WHY** it matters for our analysis.

---
### Requirements Coverage
| Requirement | Minimum | Queries |
|---|---|---|
| Total queries | 10 | **12** |
| JOINs | 3 | Q1–Q12 (all) |
| Window functions | 2 | Q7, Q8, Q9 |
| GROUP BY / ROLLUP / CUBE | 2 | Q1, Q2, Q3 |
| Subqueries | 2 | Q10, Q11, Q12 |

---
## Setup: Load Data into SQLite

In [1]:
import pandas as pd
import sqlite3

# Load the exported CSV from our main analysis
df = pd.read_csv('middle_income_trap_data.csv')

conn = sqlite3.connect('middle_income_trap.db')

# ── Table 1: Country metadata ──
countries_df = df[['country', 'group']].drop_duplicates().rename(columns={'group': 'group_label'})
countries_df.to_sql('countries', conn, if_exists='replace', index=False)

# ── Table 2: Core economic indicators ──
econ_cols = ['country', 'year', 'gdp_per_capita', 'gdp_per_capita_const', 'gdp_growth',
             'industry_pct_gdp', 'manufacturing_pct_gdp', 'services_pct_gdp',
             'education_expenditure', 'tertiary_enrollment', 'exports_pct_gdp',
             'inflation_rate', 'unemployment_rate', 'fdi_pct_gdp',
             'govt_debt_pct_gdp', 'gross_savings_pct_gni', 'population_growth']
df[econ_cols].to_sql('econ_indicators', conn, if_exists='replace', index=False)

# ── Table 3: Governance scores ──
gov_cols = ['country', 'year', 'control_of_corruption', 'rule_of_law',
            'govt_effectiveness', 'regulatory_quality']
df[gov_cols].dropna(subset=gov_cols[2:], how='all').to_sql('governance', conn, if_exists='replace', index=False)

# ── Table 4: Innovation indicators ──
innov_cols = ['country', 'year', 'rd_pct_gdp', 'patent_applications', 'hightech_exports_pct']
df[innov_cols].dropna(subset=innov_cols[2:], how='all').to_sql('innovation', conn, if_exists='replace', index=False)

# Verify table sizes
for table in ['countries', 'econ_indicators', 'governance', 'innovation']:
    n = pd.read_sql(f'SELECT COUNT(*) as n FROM {table}', conn).iloc[0, 0]
    print(f'  {table:<20s}: {n:>4d} rows')

  countries           :    7 rows
  econ_indicators     :  308 rows
  governance          :  175 rows
  innovation          :  308 rows


In [2]:
def run_query(sql, conn=conn):
    """Run a SQL query and display the result as a DataFrame."""
    result = pd.read_sql(sql, conn)
    print(result.to_string(index=False))
    return result

---
## Query 1: GROUP BY — Average GDP per capita by group and decade
- **WHAT:** Calculates mean GDP per capita for Success vs Trapped countries by decade.
- **HOW:** Groups by decade (derived via integer division) and group_label; aggregates with AVG.
- **WHY:** To see when the two groups began to diverge — the core question of our analysis.
- **Finding:** The gap was ~$1,500 in the 1980s but widened to ~$15,000 by the 2020s.

In [3]:
run_query('''
    SELECT
        CAST(e.year / 10 AS INT) * 10 AS decade,
        c.group_label,
        ROUND(AVG(e.gdp_per_capita), 0)       AS avg_gdp_pc,
        ROUND(AVG(e.gdp_per_capita_const), 0)  AS avg_gdp_pc_const,
        ROUND(AVG(e.gdp_growth), 2)            AS avg_growth
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    GROUP BY decade, c.group_label
    ORDER BY decade, c.group_label
''')

 decade group_label  avg_gdp_pc  avg_gdp_pc_const  avg_growth
   1980     Success      2501.0            5326.0        6.30
   1980     Trapped      1999.0            5288.0        3.69
   1990     Success      5926.0            8602.0        5.89
   1990     Trapped      3583.0            5648.0        3.00
   2000     Success     11159.0           13432.0        4.38
   2000     Trapped      5424.0            6628.0        3.15
   2010     Success     19403.0           18577.0        3.53
   2010     Trapped      8561.0            7660.0        2.29
   2020     Success     23273.0           21803.0        2.21
   2020     Trapped      8266.0            7656.0        0.88


,decade,group_label,avg_gdp_pc,avg_gdp_pc_const,avg_growth
0,1980,Success,2501.0,5326.0,6.30
1,1980,Trapped,1999.0,5288.0,3.69
2,1990,Success,5926.0,8602.0,5.89
3,1990,Trapped,3583.0,5648.0,3.00
4,2000,Success,11159.0,13432.0,4.38
5,2000,Trapped,5424.0,6628.0,3.15
6,2010,Success,19403.0,18577.0,3.53
7,2010,Trapped,8561.0,7660.0,2.29
8,2020,Success,23273.0,21803.0,2.21
9,2020,Trapped,8266.0,7656.0,0.88


---
## Query 2: GROUP BY with ROLLUP simulation — Indicator summary with grand total
- **WHAT:** Produces group-level averages for key indicators WITH a grand total row.
- **HOW:** Uses UNION ALL to simulate ROLLUP — first query groups by group_label, second computes overall averages with group_label = 'ALL'.
- **WHY:** Lets us compare group averages while also seeing the overall dataset average in a single result set.
- **Finding:** Success countries average $11,672 GDP/capita vs $5,199 for Trapped (overall: $7,849).

In [4]:
run_query('''
    SELECT
        c.group_label,
        COUNT(*)                                    AS n_obs,
        ROUND(AVG(e.gdp_per_capita), 0)             AS avg_gdp_pc,
        ROUND(AVG(e.gdp_growth), 2)                 AS avg_growth,
        ROUND(AVG(e.tertiary_enrollment), 1)         AS avg_tertiary,
        ROUND(AVG(e.manufacturing_pct_gdp), 1)       AS avg_manuf,
        ROUND(AVG(e.exports_pct_gdp), 1)             AS avg_exports
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    GROUP BY c.group_label

    UNION ALL

    SELECT
        \'ALL\' AS group_label,
        COUNT(*)                                    AS n_obs,
        ROUND(AVG(e.gdp_per_capita), 0)             AS avg_gdp_pc,
        ROUND(AVG(e.gdp_growth), 2)                 AS avg_growth,
        ROUND(AVG(e.tertiary_enrollment), 1)         AS avg_tertiary,
        ROUND(AVG(e.manufacturing_pct_gdp), 1)       AS avg_manuf,
        ROUND(AVG(e.exports_pct_gdp), 1)             AS avg_exports
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
''')

group_label  n_obs  avg_gdp_pc  avg_growth  avg_tertiary  avg_manuf  avg_exports
    Success    132     11672.0        4.63          78.5       19.3         34.2
    Trapped    176      5199.0        2.84          35.2       20.6         28.4
        ALL    308      7849.0        3.57          55.9       20.1         30.7


,group_label,n_obs,avg_gdp_pc,avg_growth,avg_tertiary,avg_manuf,avg_exports
0,Success,132,11672.0,4.63,78.5,19.3,34.2
1,Trapped,176,5199.0,2.84,35.2,20.6,28.4
2,ALL,308,7849.0,3.57,55.9,20.1,30.7


---
## Query 3: GROUP BY with CUBE simulation — Multi-dimensional summary
- **WHAT:** Average GDP growth by group AND income classification, with all subtotals.
- **HOW:** Uses UNION ALL to simulate CUBE — produces group×income, group-only, income-only, and grand total rows.
- **WHY:** To understand if growth patterns differ not just by group, but by income stage within each group.
- **Finding:** Trapped countries grow slower even at the same income level.

In [5]:
run_query('''
    -- Full detail: group × income_class
    SELECT
        c.group_label,
        CASE
            WHEN e.gdp_per_capita < 1136  THEN \'Low\'
            WHEN e.gdp_per_capita < 4466  THEN \'Lower-Middle\'
            WHEN e.gdp_per_capita < 13845 THEN \'Upper-Middle\'
            ELSE \'High\'
        END AS income_class,
        COUNT(*) AS n_obs,
        ROUND(AVG(e.gdp_growth), 2) AS avg_growth
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.gdp_per_capita IS NOT NULL
    GROUP BY c.group_label, income_class

    UNION ALL

    -- Subtotal: by group only
    SELECT
        c.group_label,
        \'ALL\' AS income_class,
        COUNT(*) AS n_obs,
        ROUND(AVG(e.gdp_growth), 2) AS avg_growth
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.gdp_per_capita IS NOT NULL
    GROUP BY c.group_label

    UNION ALL

    -- Grand total
    SELECT
        \'ALL\' AS group_label,
        \'ALL\' AS income_class,
        COUNT(*) AS n_obs,
        ROUND(AVG(e.gdp_growth), 2) AS avg_growth
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.gdp_per_capita IS NOT NULL

    ORDER BY group_label, income_class
''')

group_label income_class  n_obs  avg_growth
        ALL          ALL    298        3.57
    Success          ALL    122        4.63
    Success         High     42        3.71
    Success Lower-Middle     33        5.44
    Success Upper-Middle     47        4.90
    Trapped          ALL    176        2.84
    Trapped         High      1        3.35
    Trapped          Low      8        5.93
    Trapped Lower-Middle     81        3.12
    Trapped Upper-Middle     86        2.27


,group_label,income_class,n_obs,avg_growth
0,ALL,ALL,298,3.57
1,Success,ALL,122,4.63
2,Success,High,42,3.71
3,Success,Lower-Middle,33,5.44
4,Success,Upper-Middle,47,4.90
5,Trapped,ALL,176,2.84
6,Trapped,High,1,3.35
7,Trapped,Low,8,5.93
8,Trapped,Lower-Middle,81,3.12
9,Trapped,Upper-Middle,86,2.27


---
## Query 4: JOIN — Economic indicators with governance scores
- **WHAT:** Joins core economic data with governance quality scores for each country-year.
- **HOW:** LEFT JOIN governance on country+year so we keep all economic rows even when governance data is unavailable (governance starts from 1996).
- **WHY:** To analyze the correlation between institutional quality and economic performance.
- **Finding:** Countries with higher governance scores consistently have higher GDP per capita.

In [6]:
run_query('''
    SELECT
        e.country,
        e.year,
        ROUND(e.gdp_per_capita, 0) AS gdp_pc,
        ROUND(e.gdp_growth, 2) AS growth,
        ROUND(g.control_of_corruption, 2) AS corruption,
        ROUND(g.rule_of_law, 2) AS rule_of_law,
        c.group_label
    FROM econ_indicators e
    JOIN countries c    ON e.country = c.country
    LEFT JOIN governance g ON e.country = g.country AND e.year = g.year
    WHERE e.year IN (2000, 2010, 2020)
    ORDER BY e.year, c.group_label, e.country
''')

     country  year  gdp_pc  growth  corruption  rule_of_law group_label
       Chile  2000  5053.0    4.97        1.54         1.26     Success
 Korea, Rep.  2000 12710.0    9.20        0.27         0.83     Success
      Poland  2000  4521.0    4.66        0.70         0.63     Success
      Brazil  2000  3767.0    4.39        0.01        -0.24     Trapped
      Mexico  2000  7524.0    5.03       -0.07        -0.43     Trapped
South Africa  2000  3218.0    4.20        0.55         0.15     Trapped
    Thailand  2000  2006.0    4.46       -0.23         0.58     Trapped
       Chile  2010 12633.0    5.85        1.44         1.23     Success
 Korea, Rep.  2010 24071.0    6.98        0.43         1.01     Success
      Poland  2010 12568.0    3.17        0.51         0.71     Success
      Brazil  2010 11403.0    7.53        0.04         0.06     Trapped
      Mexico  2010  9729.0    4.97       -0.44        -0.55     Trapped
South Africa  2010  7973.0    3.04        0.07         0.10     

,country,year,gdp_pc,growth,corruption,rule_of_law,group_label
0,Chile,2000,5053.0,4.97,1.54,1.26,Success
1,"Korea, Rep.",2000,12710.0,9.20,0.27,0.83,Success
2,Poland,2000,4521.0,4.66,0.70,0.63,Success
3,Brazil,2000,3767.0,4.39,0.01,-0.24,Trapped
4,Mexico,2000,7524.0,5.03,-0.07,-0.43,Trapped
5,South Africa,2000,3218.0,4.20,0.55,0.15,Trapped
6,Thailand,2000,2006.0,4.46,-0.23,0.58,Trapped
7,Chile,2010,12633.0,5.85,1.44,1.23,Success
8,"Korea, Rep.",2010,24071.0,6.98,0.43,1.01,Success
9,Poland,2010,12568.0,3.17,0.51,0.71,Success


---
## Query 5: Three-way JOIN — Economic + Governance + Innovation
- **WHAT:** Combines all three external datasets for a comprehensive country-year profile.
- **HOW:** Three-way LEFT JOIN so we keep all rows even if governance or innovation is NULL.
- **WHY:** To build the comprehensive view that feeds our regression models.
- **Finding:** Korea leads in all dimensions; trapped countries lag most in R&D and governance.

In [7]:
run_query('''
    SELECT
        e.country,
        c.group_label,
        e.year,
        ROUND(e.gdp_per_capita, 0) AS gdp_pc,
        ROUND(e.gdp_growth, 2) AS growth,
        ROUND(g.rule_of_law, 2) AS rule_of_law,
        ROUND(i.rd_pct_gdp, 3) AS rd_pct_gdp,
        ROUND(i.patent_applications, 0) AS patents
    FROM econ_indicators e
    JOIN countries c       ON e.country = c.country
    LEFT JOIN governance g ON e.country = g.country AND e.year = g.year
    LEFT JOIN innovation i ON e.country = i.country AND i.year = e.year
    WHERE e.year IN (2005, 2015)
    ORDER BY e.year, c.group_label, e.country
''')

     country group_label  year  gdp_pc  growth  rule_of_law  rd_pct_gdp  patents
       Chile     Success  2005  7480.0    5.84         1.21         NaN    361.0
 Korea, Rep.     Success  2005 20167.0    4.36         0.97       2.523 122188.0
      Poland     Success  2005  8044.0    3.26         0.47       0.563   2028.0
      Brazil     Trapped  2005  4828.0    3.20        -0.48       1.002   4054.0
      Mexico     Trapped  2005  8672.0    2.11        -0.38       0.381    584.0
South Africa     Trapped  2005  5837.0    5.28        -0.01       0.770   1003.0
    Thailand     Trapped  2005  2868.0    4.19        -0.01       0.219    891.0
       Chile     Success  2015 13434.0    2.15         1.22       0.383    443.0
 Korea, Rep.     Success  2015 30172.0    2.92         0.90       3.978 167275.0
      Poland     Success  2015 12638.0    4.43         0.80       1.004   4676.0
      Brazil     Trapped  2015  8936.0   -3.55        -0.19       1.371   4641.0
      Mexico     Trapped  20

,country,group_label,year,gdp_pc,growth,rule_of_law,rd_pct_gdp,patents
0,Chile,Success,2005,7480.0,5.84,1.21,NaN,361.0
1,"Korea, Rep.",Success,2005,20167.0,4.36,0.97,2.523,122188.0
2,Poland,Success,2005,8044.0,3.26,0.47,0.563,2028.0
3,Brazil,Trapped,2005,4828.0,3.20,-0.48,1.002,4054.0
4,Mexico,Trapped,2005,8672.0,2.11,-0.38,0.381,584.0
5,South Africa,Trapped,2005,5837.0,5.28,-0.01,0.770,1003.0
6,Thailand,Trapped,2005,2868.0,4.19,-0.01,0.219,891.0
7,Chile,Success,2015,13434.0,2.15,1.22,0.383,443.0
8,"Korea, Rep.",Success,2015,30172.0,2.92,0.90,3.978,167275.0
9,Poland,Success,2015,12638.0,4.43,0.80,1.004,4676.0


---
## Query 6: JOIN + GROUP BY — Average governance by group
- **WHAT:** Compares institutional quality between Success and Trapped groups.
- **HOW:** Joins governance with countries, groups by group_label, aggregates with AVG.
- **WHY:** Governance quality emerged as the most consistent differentiator in our EDA.
- **Finding:** Success countries score +0.8 to +1.1 points higher across all four governance dimensions.

In [8]:
run_query('''
    SELECT
        c.group_label,
        ROUND(AVG(g.control_of_corruption), 3) AS avg_corruption,
        ROUND(AVG(g.rule_of_law), 3)           AS avg_rule_of_law,
        ROUND(AVG(g.govt_effectiveness), 3)    AS avg_govt_effect,
        ROUND(AVG(g.regulatory_quality), 3)    AS avg_reg_quality
    FROM governance g
    JOIN countries c ON g.country = c.country
    GROUP BY c.group_label
    ORDER BY c.group_label
''')

group_label  avg_corruption  avg_rule_of_law  avg_govt_effect  avg_reg_quality
    Success           0.771            0.895            0.841            1.020
    Trapped          -0.266           -0.182            0.078            0.178


,group_label,avg_corruption,avg_rule_of_law,avg_govt_effect,avg_reg_quality
0,Success,0.771,0.895,0.841,1.020
1,Trapped,-0.266,-0.182,0.078,0.178


---
## Query 7: WINDOW FUNCTION — Rank countries by GDP per capita each year
- **WHAT:** Ranks all 7 countries by GDP per capita within each year.
- **HOW:** Uses RANK() window function partitioned by year, ordered by GDP descending.
- **WHY:** To track how rankings shift over time — did Korea always lead?
- **Finding:** Korea was ranked 3rd in 1980 but rose to 1st by 2000 and stayed there.

In [9]:
run_query('''
    SELECT
        e.country,
        c.group_label,
        e.year,
        ROUND(e.gdp_per_capita, 0) AS gdp_pc,
        RANK() OVER (PARTITION BY e.year ORDER BY e.gdp_per_capita DESC) AS gdp_rank
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.gdp_per_capita IS NOT NULL
      AND e.year IN (1980, 1990, 2000, 2010, 2020, 2023)
    ORDER BY e.year, gdp_rank
''')

     country group_label  year  gdp_pc  gdp_rank
      Mexico     Trapped  1980  3055.0         1
South Africa     Trapped  1980  3029.0         2
       Chile     Success  1980  2571.0         3
      Brazil     Trapped  1980  1959.0         4
 Korea, Rep.     Success  1980  1746.0         5
    Thailand     Trapped  1980   709.0         6
 Korea, Rep.     Success  1990  6813.0         1
      Mexico     Trapped  1990  3154.0         2
South Africa     Trapped  1990  3093.0         3
      Brazil     Trapped  1990  2620.0         4
       Chile     Success  1990  2488.0         5
      Poland     Success  1990  1731.0         6
    Thailand     Trapped  1990  1559.0         7
 Korea, Rep.     Success  2000 12710.0         1
      Mexico     Trapped  2000  7524.0         2
       Chile     Success  2000  5053.0         3
      Poland     Success  2000  4521.0         4
      Brazil     Trapped  2000  3767.0         5
South Africa     Trapped  2000  3218.0         6
    Thailand     Tra

,country,group_label,year,gdp_pc,gdp_rank
0,Mexico,Trapped,1980,3055.0,1
1,South Africa,Trapped,1980,3029.0,2
2,Chile,Success,1980,2571.0,3
3,Brazil,Trapped,1980,1959.0,4
4,"Korea, Rep.",Success,1980,1746.0,5
5,Thailand,Trapped,1980,709.0,6
6,"Korea, Rep.",Success,1990,6813.0,1
7,Mexico,Trapped,1990,3154.0,2
8,South Africa,Trapped,1990,3093.0,3
9,Brazil,Trapped,1990,2620.0,4


---
## Query 8: WINDOW FUNCTION — Year-over-year GDP growth with LAG
- **WHAT:** Calculates YoY change in GDP per capita using LAG.
- **HOW:** LAG(gdp_per_capita_const, 1) gets the prior year's value per country, then we compute percent change.
- **WHY:** Validates our Python-computed gdp_pc_yoy_growth feature and shows SQL can replicate pandas pct_change().
- **Finding:** Korea's YoY growth is consistently positive; Brazil and South Africa show frequent negative years.

In [10]:
run_query('''
    SELECT
        e.country,
        c.group_label,
        e.year,
        ROUND(e.gdp_per_capita_const, 0) AS gdp_pc_const,
        ROUND(LAG(e.gdp_per_capita_const, 1) OVER (
            PARTITION BY e.country ORDER BY e.year
        ), 0) AS prev_year_gdp,
        ROUND(
            (e.gdp_per_capita_const - LAG(e.gdp_per_capita_const, 1) OVER (
                PARTITION BY e.country ORDER BY e.year
            )) * 100.0 / LAG(e.gdp_per_capita_const, 1) OVER (
                PARTITION BY e.country ORDER BY e.year
            ), 2
        ) AS yoy_growth_pct
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.year BETWEEN 2018 AND 2023
    ORDER BY e.country, e.year
''')

     country group_label  year  gdp_pc_const  prev_year_gdp  yoy_growth_pct
      Brazil     Trapped  2018        8722.0            NaN             NaN
      Brazil     Trapped  2019        8771.0         8722.0            0.56
      Brazil     Trapped  2020        8435.0         8771.0           -3.84
      Brazil     Trapped  2021        8799.0         8435.0            4.32
      Brazil     Trapped  2022        9032.0         8799.0            2.65
      Brazil     Trapped  2023        9288.0         9032.0            2.83
       Chile     Success  2018       13763.0            NaN             NaN
       Chile     Success  2019       13631.0        13763.0           -0.96
       Chile     Success  2020       12679.0        13631.0           -6.98
       Chile     Success  2021       14051.0        12679.0           10.82
       Chile     Success  2022       14283.0        14051.0            1.65
       Chile     Success  2023       14280.0        14283.0           -0.02
 Korea, Rep.

,country,group_label,year,gdp_pc_const,prev_year_gdp,yoy_growth_pct
0,Brazil,Trapped,2018,8722.0,NaN,NaN
1,Brazil,Trapped,2019,8771.0,8722.0,0.56
2,Brazil,Trapped,2020,8435.0,8771.0,-3.84
3,Brazil,Trapped,2021,8799.0,8435.0,4.32
4,Brazil,Trapped,2022,9032.0,8799.0,2.65
5,Brazil,Trapped,2023,9288.0,9032.0,2.83
6,Chile,Success,2018,13763.0,NaN,NaN
7,Chile,Success,2019,13631.0,13763.0,-0.96
8,Chile,Success,2020,12679.0,13631.0,-6.98
9,Chile,Success,2021,14051.0,12679.0,10.82


---
## Query 9: WINDOW FUNCTION — Running average of R&D spending
- **WHAT:** Computes a 5-year moving average of R&D expenditure per country.
- **HOW:** Uses AVG() with a ROWS BETWEEN window frame, partitioned by country.
- **WHY:** R&D spending is volatile year-to-year; the moving average reveals long-term trends.
- **Finding:** Korea's R&D trending steadily upward to 4.5%+; trapped countries flat around 0.5-1%.

In [11]:
run_query('''
    SELECT
        i.country,
        c.group_label,
        i.year,
        ROUND(i.rd_pct_gdp, 3) AS rd_pct_gdp,
        ROUND(AVG(i.rd_pct_gdp) OVER (
            PARTITION BY i.country
            ORDER BY i.year
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ), 3) AS rd_5yr_avg
    FROM innovation i
    JOIN countries c ON i.country = c.country
    WHERE i.rd_pct_gdp IS NOT NULL
      AND i.year >= 2010
    ORDER BY c.group_label, i.country, i.year
''')

     country group_label  year  rd_pct_gdp  rd_5yr_avg
       Chile     Success  2010       0.332       0.332
       Chile     Success  2011       0.353       0.342
       Chile     Success  2012       0.362       0.349
       Chile     Success  2013       0.390       0.359
       Chile     Success  2014       0.377       0.363
       Chile     Success  2015       0.383       0.373
       Chile     Success  2016       0.371       0.377
       Chile     Success  2017       0.357       0.376
       Chile     Success  2018       0.369       0.371
       Chile     Success  2019       0.342       0.364
       Chile     Success  2020       0.335       0.355
       Chile     Success  2021       0.360       0.353
 Korea, Rep.     Success  2010       3.316       3.316
 Korea, Rep.     Success  2011       3.592       3.454
 Korea, Rep.     Success  2012       3.850       3.586
 Korea, Rep.     Success  2013       3.951       3.677
 Korea, Rep.     Success  2014       4.078       3.757
 Korea, Re

,country,group_label,year,rd_pct_gdp,rd_5yr_avg
0,Chile,Success,2010,0.332,0.332
1,Chile,Success,2011,0.353,0.342
2,Chile,Success,2012,0.362,0.349
3,Chile,Success,2013,0.390,0.359
4,Chile,Success,2014,0.377,0.363
...,...,...,...,...,...
82,Thailand,Trapped,2018,1.114,0.798
83,Thailand,Trapped,2019,1.143,0.931
84,Thailand,Trapped,2020,1.328,1.073
85,Thailand,Trapped,2021,1.208,1.159


---
## Query 10: SUBQUERY (correlated) — Countries outperforming their group average
- **WHAT:** Finds country-years where GDP growth exceeds the group average for that year.
- **HOW:** Uses a correlated subquery to compute the group average, then filters rows that beat it.
- **WHY:** Identifies which countries consistently "pull up" or "drag down" their group.
- **Finding:** Chile and Thailand are frequent outperformers within their respective groups.

In [12]:
run_query('''
    SELECT
        e.country,
        c.group_label,
        e.year,
        ROUND(e.gdp_growth, 2) AS country_growth,
        ROUND((
            SELECT AVG(e2.gdp_growth)
            FROM econ_indicators e2
            JOIN countries c2 ON e2.country = c2.country
            WHERE c2.group_label = c.group_label
              AND e2.year = e.year
        ), 2) AS group_avg_growth
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.gdp_growth > (
        SELECT AVG(e2.gdp_growth)
        FROM econ_indicators e2
        JOIN countries c2 ON e2.country = c2.country
        WHERE c2.group_label = c.group_label
          AND e2.year = e.year
    )
    AND e.year >= 2015
    ORDER BY e.year, c.group_label, e.country
''')

     country group_label  year  country_growth  group_avg_growth
      Poland     Success  2015            4.43              3.17
      Mexico     Trapped  2015            2.70              0.90
South Africa     Trapped  2015            1.32              0.90
    Thailand     Trapped  2015            3.13              0.90
 Korea, Rep.     Success  2016            3.17              2.65
      Poland     Success  2016            3.03              2.65
      Mexico     Trapped  2016            1.77              0.65
South Africa     Trapped  2016            0.66              0.65
    Thailand     Trapped  2016            3.44              0.65
 Korea, Rep.     Success  2017            3.43              3.31
      Poland     Success  2017            5.15              3.31
    Thailand     Trapped  2017            4.18              2.13
      Poland     Success  2018            6.25              4.47
    Thailand     Trapped  2018            4.22              2.38
      Poland     Success 

,country,group_label,year,country_growth,group_avg_growth
0,Poland,Success,2015,4.43,3.17
1,Mexico,Trapped,2015,2.70,0.90
2,South Africa,Trapped,2015,1.32,0.90
3,Thailand,Trapped,2015,3.13,0.90
4,"Korea, Rep.",Success,2016,3.17,2.65
5,Poland,Success,2016,3.03,2.65
6,Mexico,Trapped,2016,1.77,0.65
7,South Africa,Trapped,2016,0.66,0.65
8,Thailand,Trapped,2016,3.44,0.65
9,"Korea, Rep.",Success,2017,3.43,3.31


---
## Query 11: SUBQUERY (CTE style) — High R&D vs Low R&D performance
- **WHAT:** Compares average GDP growth for high-R&D vs low-R&D country-years.
- **HOW:** A CTE (WITH clause) computes median R&D, then the outer query classifies and aggregates.
- **WHY:** Tests our hypothesis that R&D investment predicts economic growth.
- **Finding:** High R&D country-years average higher GDP and are almost exclusively in the Success group.

In [13]:
run_query('''
    WITH rd_stats AS (
        SELECT AVG(rd_pct_gdp) AS mean_rd
        FROM innovation
        WHERE rd_pct_gdp IS NOT NULL
    )
    SELECT
        CASE WHEN i.rd_pct_gdp >= s.mean_rd THEN \'High R&D\' ELSE \'Low R&D\' END AS rd_group,
        c.group_label,
        COUNT(*)                        AS n_obs,
        ROUND(AVG(e.gdp_growth), 2)     AS avg_growth,
        ROUND(AVG(e.gdp_per_capita), 0) AS avg_gdp_pc,
        ROUND(AVG(i.rd_pct_gdp), 3)     AS avg_rd
    FROM innovation i
    JOIN econ_indicators e ON i.country = e.country AND i.year = e.year
    JOIN countries c       ON i.country = c.country
    CROSS JOIN rd_stats s
    WHERE i.rd_pct_gdp IS NOT NULL
    GROUP BY rd_group, c.group_label
    ORDER BY rd_group, c.group_label
''')

rd_group group_label  n_obs  avg_growth  avg_gdp_pc  avg_rd
High R&D     Success     32        4.14     22428.0   3.002
High R&D     Trapped     18        1.14      9364.0   1.188
 Low R&D     Success     37        3.64     11081.0   0.569
 Low R&D     Trapped     77        2.69      6487.0   0.514


,rd_group,group_label,n_obs,avg_growth,avg_gdp_pc,avg_rd
0,High R&D,Success,32,4.14,22428.0,3.002
1,High R&D,Trapped,18,1.14,9364.0,1.188
2,Low R&D,Success,37,3.64,11081.0,0.569
3,Low R&D,Trapped,77,2.69,6487.0,0.514


---
## Query 12: SUBQUERY (NOT IN) — Countries that never reached high-income status
- **WHAT:** Lists countries that never crossed the high-income threshold ($13,845).
- **HOW:** Uses a NOT IN subquery to exclude countries that have ANY year above the threshold.
- **WHY:** Directly identifies which countries remain "trapped" by World Bank income classification.
- **Finding:** Brazil, South Africa, and Thailand have never reached high-income status.

In [14]:
run_query('''
    SELECT
        e.country,
        c.group_label,
        ROUND(MAX(e.gdp_per_capita), 0) AS peak_gdp_pc,
        MAX(e.year) AS latest_year
    FROM econ_indicators e
    JOIN countries c ON e.country = c.country
    WHERE e.country NOT IN (
        SELECT DISTINCT e2.country
        FROM econ_indicators e2
        WHERE e2.gdp_per_capita >= 13845
    )
    GROUP BY e.country, c.group_label
    ORDER BY peak_gdp_pc DESC
''')

     country group_label  peak_gdp_pc  latest_year
      Brazil     Trapped      13397.0         2023
South Africa     Trapped       8646.0         2023
    Thailand     Trapped       7606.0         2023


,country,group_label,peak_gdp_pc,latest_year
0,Brazil,Trapped,13397.0,2023
1,South Africa,Trapped,8646.0,2023
2,Thailand,Trapped,7606.0,2023


---
## Cleanup

In [15]:
conn.close()
print('Database connection closed.')

Database connection closed.
